In [3]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

#file yang mau diambil
file_path = "data.csv"

#load API (harus pake ISO)
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "carrie1/ecommerce-data",
    file_path,
    pandas_kwargs={"encoding":"ISO-8859-1"}
)

#check data
print("Hasil Extract")
print(df.head())
print("\nJumlah baris data:", len(df))

/var/folders/j2/kvcnccl12rn8s2jnbgrvn_g40000gn/T/ipykernel_90082/3890813717.py:8: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████████████████████████████████| 7.20M/7.20M [00:04<00:00, 1.64MB/s]

Extracting zip of data.csv...


Hasil Extract
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2010 8:26       2.55     17850.0  United Kingdom  
1  12/1/2010 8:26       3.39     17850.0  United Kingdom  
2  12/1/2010 8:26       2.75     17850.0  United Kingdom  
3  12/1/2010 8:26       3.39     17850.0  United Kingdom  
4  12/1/2010 8:26       3.39     17850.0  United Kingdom  

Jumlah baris data: 541909


In [5]:
#explor data
#check ringkasan info data (tipe data & jumlah null)
print("---Info Dataset---")
df.info()

#check missing values
print("\n---Jumlah Data Kosong---")
print(df.isnull().sum())

#duplicate data
print(f"\nJumlah baris duplikat: {df.duplicated().sum()}")

---Info Dataset---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB

---Jumlah Data Kosong---
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Jumlah baris duplikat: 5268


In [10]:
#cleaning & transformasi 
import pandas as pd
#buat salinan
df_clean = df.copy()

#hapus customerID yang kosong
df_clean = df_clean.dropna(subset=['CustomerID'])

#Ubah tipe data CustomerID
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

#Ubah tipe data InvoiceDate
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

#filter data
df_clean = df_clean[~df_clean['InvoiceNo'].str.contains('C', na=False)]

#tambah kolom feature engineering
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

#hapus duplicate
df_clean = df_clean.drop_duplicates()

print("berhasil dibersihkan")

berhasil dibersihkan


In [11]:
import sqlite3

# Langkah 1: Bikin koneksi (ini otomatis bikin file kalau belum ada)
conn = sqlite3.connect('ecommerce.db')

# Langkah 2: Masukin data ke tabel 'sales_data'
# Pastikan variabel 'df_clean' sudah ada isinya ya
df_clean.to_sql('sales_data', conn, if_exists='replace', index=False, chunksize=10000)

# Langkah 3: WAJIB ditutup koneksinya biar file-nya "ke-save" di laptop
conn.close()

print("File ecommerce.db harusnya sudah muncul sekarang!")

File ecommerce.db harusnya sudah muncul sekarang!
